# 🧠 Bounding Box Coordinate Systems: Formats, Math, and Conversions

Welcome to the hands-on explanation notebook for **Bounding Box Coordinate Systems**! In this notebook, we will:
1. Explain Corners (XYXY) vs. Centroid (XYWH) representations.
2. Implement coordinate conversion functions from scratch in NumPy.
3. Formulate the mathematics of scale-normalization used in YOLO annotation text files.
4. Run a full round-trip coordinate pipeline: Raw Corners -> Raw Centroid -> YOLO Normalized -> Raw Corners.
5. Visualise a bounding box on a coordinate plane, labeling its corners, width, height, and center point.
6. Connect normalized formats to multi-scale data augmentations in YOLO training.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Set seed for reproducibility
np.random.seed(42)

## 1. Implementing Conversion Functions in NumPy

We implement vectorised conversions that handle single boxes or multiple boxes (arrays of shape $(N, 4)$).

In [ ]:
def xywh_to_xyxy(boxes):
    boxes = np.array(boxes)
    converted = np.zeros_like(boxes, dtype=float)
    converted[..., 0] = boxes[..., 0] - (boxes[..., 2] / 2.0)
    converted[..., 1] = boxes[..., 1] - (boxes[..., 3] / 2.0)
    converted[..., 2] = boxes[..., 0] + (boxes[..., 2] / 2.0)
    converted[..., 3] = boxes[..., 1] + (boxes[..., 3] / 2.0)
    return converted

def xyxy_to_xywh(boxes):
    boxes = np.array(boxes)
    converted = np.zeros_like(boxes, dtype=float)
    converted[..., 0] = (boxes[..., 0] + boxes[..., 2]) / 2.0
    converted[..., 1] = (boxes[..., 1] + boxes[..., 3]) / 2.0
    converted[..., 2] = boxes[..., 2] - boxes[..., 0]
    converted[..., 3] = boxes[..., 3] - boxes[..., 1]
    return converted

def normalize_xywh(boxes_xywh, width, height):
    boxes_xywh = np.array(boxes_xywh, dtype=float)
    normed = np.zeros_like(boxes_xywh)
    normed[..., 0] = boxes_xywh[..., 0] / width
    normed[..., 1] = boxes_xywh[..., 1] / height
    normed[..., 2] = boxes_xywh[..., 2] / width
    normed[..., 3] = boxes_xywh[..., 3] / height
    return normed

def denormalize_xywh(boxes_norm, width, height):
    boxes_norm = np.array(boxes_norm, dtype=float)
    denormed = np.zeros_like(boxes_norm)
    denormed[..., 0] = boxes_norm[..., 0] * width
    denormed[..., 1] = boxes_norm[..., 1] * height
    denormed[..., 2] = boxes_norm[..., 2] * width
    denormed[..., 3] = boxes_norm[..., 3] * height
    return denormed

## 2. Round-Trip Verification

We simulate a $640 \times 480$ pixel resolution image containing an object at corner coordinates $[100, 150, 300, 400]$:
-   We convert corners (XYXY) to centroid (XYWH).
-   We normalize the centroid to YOLO format.
-   We denormalize back to pixels.
-   We convert back to corners (XYXY) and verify alignment.

In [ ]:
W, H = 640, 480
box_xyxy = [100, 150, 300, 400]

box_xywh = xyxy_to_xywh(box_xyxy)
print("Centroid Format (XYWH)   :", box_xywh)

box_norm = normalize_xywh(box_xywh, W, H)
print("Normalized YOLO Format    :", box_norm)

box_denorm = denormalize_xywh(box_norm, W, H)

box_restored = xywh_to_xyxy(box_denorm)
print("Restored Corners (XYXY)   :", box_restored)
print("Reconstructed perfectly?  :", np.allclose(box_xyxy, box_restored))

## 3. Visualizing Bounding Box Parameters

Let's draw the bounding box inside a coordinate canvas of size $640 \times 480$, labeling the points and axes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.set_xlim(0, W)
ax.set_ylim(H, 0)

x1, y1, x2, y2 = box_xyxy
xc, yc, w, h = box_xywh
rect = patches.Rectangle((x1, y1), w, h, linewidth=3, edgecolor='green', facecolor='none')
ax.add_patch(rect)

ax.scatter(xc, yc, color='red', s=120, zorder=5, label=f'Centroid (xc={xc}, yc={yc})')

ax.text(x1 - 10, y1 - 15, f'Top-Left Corners\n(x1={x1}, y1={y1})', color='darkgreen', fontsize=10, ha='right')
ax.text(x2 + 10, y2 + 25, f'Bottom-Right Corners\n(x2={x2}, y2={y2})', color='darkgreen', fontsize=10, ha='left')
ax.text(xc + 15, yc - 15, f'w={w}, h={h}', color='blue', fontsize=11, fontweight='bold')

plt.title('Bounding Box Geometry (Image Pixel Space)')
plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

Look at the plot:
- The top-left corner $(x_1, y_1)$ is at $(100, 150)$.
- The bottom-right corner $(x_2, y_2)$ is at $(300, 400)$.
- The width is $200$ pixels, height is $250$ pixels, and the center lies at $(200, 275)$.
- The y-axis starts at 0 at the top, which matches digital image indexing.

## 💡 Connection to YOLO and Deep Learning
*   **Scale Invariance:** Why does YOLO use normalized values (`0` to `1`) inside label text files instead of pixel coordinates? During training, YOLO applies multiple augmentations, such as scaling and resizing image resolutions (e.g. training at 640x640, but running validation at 320x320).
*   If annotations used raw pixel values, we would need to scale coordinates manually for every augmentation. Because normalized values are independent of canvas size, YOLO simply multiplies the label floats by the current image dimensions, making scale scaling automatic and robust!